# S1 · Laboratorio: tu primer ETL en Databricks

**Extracción, Transformación y Carga (ETL)** · CUNEF Universidad · Sesión 1 (Parte B)

**Objetivos**
- Moverte por el workspace de Databricks Free Edition (Catalog, notebooks, SQL).
- Explorar ficheros de un volumen de Unity Catalog con Python.
- Construir un mini-ETL **extraer → transformar → cargar** con pandas y guardar el resultado como **tabla Delta**.
- Consultar la tabla con SQL y descubrir qué es una tabla Delta (`DESCRIBE DETAIL`, `DESCRIBE HISTORY`).

| | |
|---|---|
| **Duración** | ≈ 1h30 (Ej. 1–2 en 30 min · Ej. 3 en 10 min · Ej. 4 en 30 min · Ej. 5 y retos en 20 min) |
| **Requisitos** | Haber ejecutado `00_Setup_Entorno` (esquema `etl_curso` y datos en el volumen `raw`) |
| **Niveles** | ★ básico · ★★ intermedio · ★★★ reto |

Completa las celdas marcadas con `# TODO` y ejecuta las celdas de comprobación (`assert`): si no muestran
ningún error, vas bien. ✅

In [0]:
# Configuración común del curso (no la cambies salvo que uses otro catálogo/esquema)
from pyspark.sql import functions as F, Window
from pyspark.sql import types as T

CATALOG = "workspace"          # catálogo por defecto de Free Edition
SCHEMA  = "etl_curso"          # tu esquema de trabajo
RAW     = f"/Volumes/{CATALOG}/{SCHEMA}/raw"   # datos de origen
OPS     = f"/Volumes/{CATALOG}/{SCHEMA}/ops"   # checkpoints, exportaciones
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
print(f"Trabajando en {CATALOG}.{SCHEMA} · RAW = {RAW}")

Trabajando en workspace.etl_curso · RAW = /Volumes/workspace/etl_curso/raw


## Ejercicio 1 · Alta en Databricks Free Edition y tour (★)

Si estás leyendo esto dentro de Databricks, ¡ya tienes la mitad hecha! Repasa el menú lateral:

| Elemento | Para qué sirve |
|---|---|
| **Workspace** | Carpetas con notebooks, ficheros y *Git folders* |
| **Catalog** | Explorador de Unity Catalog: catálogos, esquemas, tablas, volúmenes |
| **SQL Editor** | Consultas SQL sobre el almacén SQL *serverless* |
| **Jobs & Pipelines** | Orquestación de notebooks (lo usaremos en S6) |
| **Compute** | En Free Edition, todo es *serverless*: no hay clústeres que configurar |

En **Catalog** abre `samples → nyctaxi → trips` y fíjate en la pestaña *Sample data*.

La siguiente celda comprueba que tu sesión funciona.

In [0]:
print("Versión de Spark:", spark.version)
display(spark.range(5).withColumn("cuadrado", F.col("id") * F.col("id")))

Versión de Spark: 4.2.0


id,cuadrado
0,0
1,1
2,4
3,9
4,16


## Ejercicio 2 · Preparar el entorno (★)

Ejecuta **todas** las celdas del notebook `00_Setup_Entorno` (si no lo has hecho ya) y vuelve aquí.
La celda siguiente comprueba que los datos están en el volumen.

In [0]:
import os
assert os.path.isdir(RAW), f"No encuentro {RAW}: ejecuta antes 00_Setup_Entorno"
print(sorted(os.listdir(RAW)))

['api_fallback', 'clientes', 'clientes_cambios', 'erp', 'eventos', 'productos', 'resenas', 'tiendas', 'ventas', 'ventas_incremental']


## Ejercicio 3 · Explorar los ficheros desde Python (★)

Los volúmenes se comportan como un sistema de ficheros normal: puedes usar `os`, `open()` o pandas.
Mira las primeras líneas de cada fuente y responde a las preguntas del final.

In [0]:
for carpeta in sorted(os.listdir(RAW)):
    ficheros = os.listdir(f"{RAW}/{carpeta}")
    print(f"{carpeta:20s} {len(ficheros):3d} fichero(s)")

api_fallback           1 fichero(s)
clientes               1 fichero(s)
clientes_cambios       1 fichero(s)
erp                    1 fichero(s)
eventos                7 fichero(s)
productos              1 fichero(s)
resenas                1 fichero(s)
tiendas                1 fichero(s)
ventas                12 fichero(s)
ventas_incremental     1 fichero(s)


In [0]:
def primeras_lineas(ruta, n=3):
    """Devuelve las n primeras líneas de un fichero de texto."""
    with open(ruta, encoding="utf-8") as fh:
        return [fh.readline().rstrip() for _ in range(n)]

for ruta in [f"{RAW}/clientes/clientes.csv", f"{RAW}/ventas/ventas_2025_01.csv"]:
    print(f"== {ruta}")
    for linea in primeras_lineas(ruta):
        print("  ", linea)

== /Volumes/workspace/etl_curso/raw/clientes/clientes.csv
   id_cliente,nombre,apellidos,email,telefono,fecha_nacimiento,fecha_alta,ciudad,codigo_postal,pais,segmento,acepta_marketing,fecha_actualizacion
   C01010,Iván,Fernández López,ifernandez852@gmail.com,(+34)669474632,1989-03-25,2024-02-21 15:57:39,A Coruña,15114,ES,Particular,No,2024-09-05 15:57:39
   C03028,Mateo,Díaz Ramos,mdiaz647@yahoo.es,0034-644-852-287,05/06/1973,2022-12-24 10:52:37,Madrid,28474,Spain,Particular,No,2023-08-06 10:52:37
== /Volumes/workspace/etl_curso/raw/ventas/ventas_2025_01.csv
   id_pedido;linea;fecha_pedido;id_cliente;id_producto;id_tienda;cantidad;precio_unitario;descuento_pct;moneda;metodo_pago
   PED-101026;1;01/01/2025 09:04;C02391;P0169;W01;1;492,90;0;EUR;PayPal
   PED-101026;2;01/01/2025 09:04;C02391;P0023;W01;5;505,90;10;EUR;PayPal


In [0]:
# Alternativa con las utilidades de Databricks
display(dbutils.fs.ls(f"{RAW}/ventas"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/etl_curso/raw/ventas/ventas_2025_01.csv,ventas_2025_01.csv,439133,1794509548000
dbfs:/Volumes/workspace/etl_curso/raw/ventas/ventas_2025_02.csv,ventas_2025_02.csv,396746,1794509548000
dbfs:/Volumes/workspace/etl_curso/raw/ventas/ventas_2025_03.csv,ventas_2025_03.csv,445182,1794509546000
dbfs:/Volumes/workspace/etl_curso/raw/ventas/ventas_2025_04.csv,ventas_2025_04.csv,457624,1794509548000
dbfs:/Volumes/workspace/etl_curso/raw/ventas/ventas_2025_05.csv,ventas_2025_05.csv,472614,1794509548000
dbfs:/Volumes/workspace/etl_curso/raw/ventas/ventas_2025_06.csv,ventas_2025_06.csv,483931,1794509549000
dbfs:/Volumes/workspace/etl_curso/raw/ventas/ventas_2025_07.csv,ventas_2025_07.csv,531652,1794509549000
dbfs:/Volumes/workspace/etl_curso/raw/ventas/ventas_2025_08.csv,ventas_2025_08.csv,394938,1794509546000
dbfs:/Volumes/workspace/etl_curso/raw/ventas/ventas_2025_09.csv,ventas_2025_09.csv,502359,1794509548000
dbfs:/Volumes/workspace/etl_curso/raw/ventas/ventas_2025_10.csv,ventas_2025_10.csv,532343,1794509547000


### ★ Ejercicio 3.1 · Responde observando los ficheros

Completa las variables con lo que has visto. Pista: abre también `ventas_2025_07.csv` y compara su cabecera
con la de enero.

In [0]:
print(primeras_lineas(f"{RAW}/ventas/ventas_2025_01.csv", 1)[0])
print(primeras_lineas(f"{RAW}/ventas/ventas_2025_07.csv", 1)[0])

id_pedido;linea;fecha_pedido;id_cliente;id_producto;id_tienda;cantidad;precio_unitario;descuento_pct;moneda;metodo_pago
id_pedido;linea;fecha_pedido;id_cliente;id_producto;id_tienda;cantidad;precio_unitario;descuento_pct;moneda;metodo_pago;canal_campania


In [0]:
N_FICHEROS_VENTAS = len(os.listdir(f"{RAW}/ventas"))    # TODO: cuántos CSV de ventas hay (calcúlalo con os.listdir)
print(os.listdir(f"{RAW}/ventas")) # os.listdir nos  devuelve la lista de  ficheros
print(N_FICHEROS_VENTAS) # con el len los cuenta 

SEP_CLIENTES = ","           # TODO: separador de clientes.csv # separacion de el fichero clientes
SEP_VENTAS = ";"             # TODO: separador de los CSV de ventas # separacion de el fichero ventas
DECIMAL_VENTAS = ","         # TODO: separador decimal de precio_unitario # ver precio unitario
COLUMNA_NUEVA_JULIO = "canal_campania"   # TODO: nombre de la columna que aparece a partir de julio

['ventas_2025_01.csv', 'ventas_2025_02.csv', 'ventas_2025_03.csv', 'ventas_2025_04.csv', 'ventas_2025_05.csv', 'ventas_2025_06.csv', 'ventas_2025_07.csv', 'ventas_2025_08.csv', 'ventas_2025_09.csv', 'ventas_2025_10.csv', 'ventas_2025_11.csv', 'ventas_2025_12.csv']
12


In [0]:
assert N_FICHEROS_VENTAS == 12
assert (SEP_CLIENTES, SEP_VENTAS, DECIMAL_VENTAS) == (",", ";", ","), "Revisa los separadores"
assert COLUMNA_NUEVA_JULIO == primeras_lineas(f"{RAW}/ventas/ventas_2025_07.csv", 1)[0].split(";")[-1]
print("✅ Ejercicio 3.1 correcto")

✅ Ejercicio 3.1 correcto


> 📝 **Para pensar**: anota tres problemas de calidad que veas a simple vista en `clientes.csv`
> (países escritos de distintas formas, emails en mayúsculas, fechas en dos formatos…).
> En la S4 los resolveremos todos; hoy sólo unos pocos.

## Ejercicio 4 · Mini-ETL de clientes: pandas → Delta (★★)

Vas a construir el ETL más sencillo posible con las tres fases:

| Fase | Qué haces | Herramienta |
|---|---|---|
| **E**xtraer | Leer `clientes.csv` **como texto** (sin que pandas adivine tipos) | `pd.read_csv(..., dtype=str)` |
| **T**ransformar | Duplicados, país ISO-2, email en minúsculas, fecha de alta, marketing booleano | pandas |
| **C**argar | Convertir a DataFrame de Spark y guardar como tabla **Delta** | `spark.createDataFrame` + `saveAsTable` |

### 4.1 Extraer (★)

In [0]:
import pandas as pd

pdf_raw = pd.read_csv(f"{RAW}/clientes/clientes.csv")   # TODO: lee f"{RAW}/clientes/clientes.csv" con pandas, todas las columnas como texto (dtype=str)
print(pdf_raw.shape)
pdf_raw.head()

(4129, 13)


,id_cliente,nombre,apellidos,email,telefono,fecha_nacimiento,fecha_alta,ciudad,codigo_postal,pais,segmento,acepta_marketing,fecha_actualizacion
0,C01010,Iván,Fernández López,ifernandez852@gmail.com,(+34)669474632,1989-03-25,2024-02-21 15:57:39,A Coruña,15114,ES,Particular,No,2024-09-05 15:57:39
1,C03028,Mateo,Díaz Ramos,mdiaz647@yahoo.es,0034-644-852-287,05/06/1973,2022-12-24 10:52:37,Madrid,28474,Spain,Particular,No,2023-08-06 10:52:37
2,C01890,Carmen,Gil Molina,cgil949@icloud.com,NaN,1959-05-19,2018-11-08 19:55:43,Madrid,28480,ES,Particular,Sí,2019-02-16 19:55:43
3,C01725,Tiago,Silva,tsilva369@yahoo.es,+351 609 934 917,1962-05-10,2023-03-29 07:03:20,Porto,4188-988,PT,Particular,NaN,2023-05-28 07:03:20
4,C00422,Beatriz,Martins,bmartins616@gmail.com,671146129,1971-02-11,2019-03-11 00:54:10,Lisboa,1283-874,Portugal,Particular,Sí,2019-09-02 00:54:10


In [0]:
assert pdf_raw.shape == (4129, 13), pdf_raw.shape
assert all(pdf_raw.dtypes == object), "Lee todas las columnas como texto (dtype=str)"
print("✅ Extracción correcta:", pdf_raw.shape)

✅ Extracción correcta: (4129, 13)


### 4.2 Perfilado rápido (★)

Antes de transformar, **mira** los datos: ¿cuántos duplicados exactos hay? ¿Qué valores toma `pais`?
¿Y `acepta_marketing`?

In [0]:
print("Duplicados exactos:", pdf_raw.duplicated().sum())

print("\nValores de pais:")
print(pdf_raw["pais"].value_counts(dropna=False).to_string())

print("\nValores de acepta_marketing:")
print(pdf_raw["acepta_marketing"].value_counts(dropna=False).to_string())

Duplicados exactos: 60

Valores de pais:
pais
ES                1585
España             555
Spain              308
PT                 243
GB                 204
ESP                165
UK                 155
 ES                149
US                 144
españa             140
Portugal           130
United Kingdom     111
USA                 93
Estados Unidos      61
Reino Unido         49
PRT                 37

Valores de acepta_marketing:
acepta_marketing
Sí       1157
No        891
1         360
true      350
false     293
0         286
si        218
NaN       198
no        182
S         119
N          75


### 4.3 Transformar (★★)

Completa la función `transformar(pdf)` para que:

1. Elimine los **duplicados exactos** (`drop_duplicates`).
2. Normalice `pais` a ISO-2 (`ES`, `PT`, `GB`, `US`): quita espacios, pasa a mayúsculas y aplica el diccionario
   `MAPA_PAIS`.
3. Pase `email` a minúsculas y sin espacios.
4. Convierta `fecha_alta` a fecha y hora (`pd.to_datetime`, formato `%Y-%m-%d %H:%M:%S`).
5. Convierta `acepta_marketing` a booleano: `Sí, si, S, 1, true` → `True`; `No, no, N, 0, false` → `False`;
   vacío → nulo. Usa el diccionario `MAPA_MARKETING` y el tipo `"boolean"` de pandas (admite nulos).

In [0]:
MAPA_PAIS = {
    "ES": "ES", "ESP": "ES", "ESPAÑA": "ES", "SPAIN": "ES",
    "PT": "PT", "PRT": "PT", "PORTUGAL": "PT",
    "GB": "GB", "UK": "GB", "UNITED KINGDOM": "GB", "REINO UNIDO": "GB",
    "US": "US", "USA": "US", "ESTADOS UNIDOS": "US",
}
MAPA_MARKETING = {"SÍ": True, "SI": True, "S": True, "1": True, "TRUE": True,
                  "NO": False, "N": False, "0": False, "FALSE": False}

def transformar(pdf):
    out = pdf.drop_duplicates().copy()
    out["pais"] = out["pais"].str.strip().str.upper().map(MAPA_PAIS) # TODO: .str.strip().str.upper() y .map(MAPA_PAIS)
    out["email"] = out["email"].str.lower().str.strip()       # TODO: minúsculas y sin espacios
    out["fecha_alta"] = pd.to_datetime(out["fecha_alta"], format="%Y-%m-%d %H:%M:%S")  # TODO: pd.to_datetime( format="%Y-%m-%d %H:%M:%S")
    out["acepta_marketing"] = out["acepta_marketing"].map(MAPA_MARKETING).astype("boolean") # TODO: normaliza el texto, .map(MAPA_MARKETING) y .astype("boolean")
    return out

pdf = transformar(pdf_raw)
print(pdf.shape)
pdf.head()

(4069, 13)


,id_cliente,nombre,apellidos,email,telefono,fecha_nacimiento,fecha_alta,ciudad,codigo_postal,pais,segmento,acepta_marketing,fecha_actualizacion
0,C01010,Iván,Fernández López,ifernandez852@gmail.com,(+34)669474632,1989-03-25,2024-02-21 15:57:39,A Coruña,15114,ES,Particular,<NA>,2024-09-05 15:57:39
1,C03028,Mateo,Díaz Ramos,mdiaz647@yahoo.es,0034-644-852-287,05/06/1973,2022-12-24 10:52:37,Madrid,28474,ES,Particular,<NA>,2023-08-06 10:52:37
2,C01890,Carmen,Gil Molina,cgil949@icloud.com,NaN,1959-05-19,2018-11-08 19:55:43,Madrid,28480,ES,Particular,<NA>,2019-02-16 19:55:43
3,C01725,Tiago,Silva,tsilva369@yahoo.es,+351 609 934 917,1962-05-10,2023-03-29 07:03:20,Porto,4188-988,PT,Particular,<NA>,2023-05-28 07:03:20
4,C00422,Beatriz,Martins,bmartins616@gmail.com,671146129,1971-02-11,2019-03-11 00:54:10,Lisboa,1283-874,PT,Particular,<NA>,2019-09-02 00:54:10


In [0]:
assert len(pdf) == 4069, f"Deberías tener 4 069 filas tras quitar 60 duplicados exactos (tienes {len(pdf)})"
assert pdf.duplicated().sum() == 0
assert set(pdf["pais"]) == {"ES", "PT", "GB", "US"}, set(pdf["pais"])
assert (pdf["email"].dropna() == pdf["email"].dropna().str.lower()).all()
assert str(pdf["fecha_alta"].dtype).startswith("datetime64")
assert str(pdf["acepta_marketing"].dtype) == "boolean"
assert pdf["acepta_marketing"].isna().sum() > 0, "Los vacíos deben quedar como nulos, no como False"
print("✅ Transformación correcta")
print(pdf["pais"].value_counts().to_string())

✅ Transformación correcta
pais
ES    2853
GB     513
PT     406
US     297


### 4.4 Cargar en una tabla Delta (★★)

1. Convierte el DataFrame de pandas en uno de Spark con `spark.createDataFrame(pdf)`.
2. Añade la columna de auditoría `_ingest_ts` con `F.current_timestamp()`.
3. Guárdalo como tabla gestionada `primer_etl_clientes` con `mode("overwrite")`.

En Databricks, `saveAsTable` crea por defecto una **tabla Delta** registrada en Unity Catalog
(`workspace.etl_curso.primer_etl_clientes`). `overwrite` hace que la carga sea **idempotente**: si repites la
ejecución, el resultado es el mismo.

In [0]:
df = spark.createDataFrame(pdf)
df = df.withColumn("_ingest_ts", F.current_timestamp())

(df.write
   .mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable("primer_etl_clientes"))

display(spark.table("primer_etl_clientes").limit(5))

id_cliente,nombre,apellidos,email,telefono,fecha_nacimiento,fecha_alta,ciudad,codigo_postal,pais,segmento,acepta_marketing,fecha_actualizacion,_ingest_ts
C01612,Noa,Pérez Fernández,nperez481@outlook.com,613068637,10/11/1999,2024-06-20T15:23:23.000Z,A Coruña,15448,ES,Particular,null,2024-12-25 15:23:23,2026-11-16T21:55:26.725Z
C01623,Pablo,Díaz Torres,pdiaz503@correo.es,null,21/07/1954,2021-07-13T01:36:12.000Z,Madrid,28304,ES,Particular,null,2021-10-25 01:36:12,2026-11-16T21:55:26.725Z
C02318,Laura,Jiménez Torres,ljimenez933@yahoo.es,(+34)638048030,1998-11-06,2021-01-13T22:03:52.000Z,A CORUÑA,15167,ES,Particular,true,2021-08-24 22:03:52,2026-11-16T21:55:26.725Z
C02143,Valeria,Sánchez Martínez,vsanchez627@correo.es,+34 655 919 152,1984-08-16,2024-06-23T07:56:04.000Z,Madrid,28269,ES,Particular,false,2025-02-20 07:56:04,2026-11-16T21:55:26.725Z
C01670,Sofía,Domínguez Pérez,sdominguez928@outlook.com,null,10/07/1988,2021-05-04T05:53:14.000Z,Madrid,28712,ES,Particular,null,2021-10-07 05:53:14,2026-11-16T21:55:26.725Z


In [0]:
tabla = spark.table("primer_etl_clientes")
tipos = dict(tabla.dtypes)
assert tabla.count() == 4069
assert "_ingest_ts" in tabla.columns
assert tipos["acepta_marketing"] == "boolean", tipos["acepta_marketing"]
assert tipos["fecha_alta"] == "timestamp", tipos["fecha_alta"]
print("✅ Tabla primer_etl_clientes creada con", tabla.count(), "filas")
tabla.printSchema()

✅ Tabla primer_etl_clientes creada con 4069 filas
root
 |-- id_cliente: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- apellidos: string (nullable = true)
 |-- email: string (nullable = true)
 |-- telefono: string (nullable = true)
 |-- fecha_nacimiento: string (nullable = true)
 |-- fecha_alta: timestamp (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- codigo_postal: string (nullable = true)
 |-- pais: string (nullable = true)
 |-- segmento: string (nullable = true)
 |-- acepta_marketing: boolean (nullable = true)
 |-- fecha_actualizacion: string (nullable = true)
 |-- _ingest_ts: timestamp (nullable = true)



## Ejercicio 5 · Consultar con SQL (★)

Las celdas que empiezan por `%sql` se ejecutan como SQL sobre el mismo esquema. También puedes copiar estas
consultas en el **SQL Editor**. Debajo de cada resultado, pulsa **+ → Visualización** para crear un gráfico.

In [0]:
%sql
-- Clientes por país después de normalizar
SELECT pais, COUNT(*) AS clientes
FROM primer_etl_clientes
GROUP BY pais
ORDER BY clientes DESC

pais,clientes
ES,2853
GB,513
PT,406
US,297


In [0]:
%sql
-- Porcentaje de clientes que aceptan marketing por segmento (¡el segmento aún está sucio!)
SELECT segmento,
       COUNT(*) AS clientes,
       ROUND(AVG(CAST(acepta_marketing AS INT)) * 100, 1) AS pct_marketing
FROM primer_etl_clientes
GROUP BY segmento
ORDER BY clientes DESC

segmento,clientes,pct_marketing
Particular,3100,57.0
Empresa,420,56.8
PARTICULAR,261,49.1
particular,224,57.4
EMPRESA,36,100.0
empresa,28,100.0


In [0]:
%sql
-- ¿En qué formato se ha guardado la tabla? ¿Cuántos ficheros tiene?
DESCRIBE DETAIL primer_etl_clientes

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,6d6a8839-9fba-429f-8409-b5bf2b77c70c,workspace.etl_curso.primer_etl_clientes,null,,2026-11-16T21:55:25.643Z,2026-11-16T21:55:33.000Z,List(),List(),4,185991,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, io.unitycatalog.tableId -> cf0b3fa4-6b61-4450-893b-61dc1bfe3d6c, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-e1abd33a-09e8-446d-9bde-d85377b9d75d, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-0f56e713-2761-4ea1-b0d3-130bacba3e76, delta.enableInCommitTimestamps -> true)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, inCommitTimestamp, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",true


### ★ Ejercicio 5.1 · De SQL a Python

`spark.sql()` devuelve un DataFrame: así combinas SQL y Python. Calcula con SQL cuántos clientes de España
(`ES`) aceptan marketing (`acepta_marketing = true`) y guarda el número en `n_es_marketing`.

In [0]:
n_es_marketing = spark.sql("SELECT COUNT(*) AS n FROM primer_etl_clientes WHERE pais = 'ES' AND acepta_marketing = true").first()["n"]
   # TODO: spark.sql("SELECT COUNT(*) AS n FROM ... WHERE ...").first()["n"]
print(n_es_marketing)

327


In [0]:
esperado = int(((pdf["pais"] == "ES") & (pdf["acepta_marketing"] == True)).sum())
assert n_es_marketing == esperado, (n_es_marketing, esperado)
print("✅ Correcto:", n_es_marketing, "clientes de España aceptan marketing")

✅ Correcto: 327 clientes de España aceptan marketing


### ★★ Ejercicio 5.2 · Historial de la tabla

Una tabla Delta guarda un **registro de transacciones** (`_delta_log`): cada escritura es una versión.
Ejecuta otra vez la celda de carga (4.4) y vuelve a lanzar esta consulta. ¿Qué cambia? ¿Cambia el número de filas?

In [0]:
%sql
DESCRIBE HISTORY primer_etl_clientes

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-11-16T21:55:32.367Z,71829128455803,ccesarmata@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""cf0b3fa4-6b61-4450-893b-61dc1bfe3d6c"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-e1abd33a-09e8-446d-9bde-d85377b9d75d"",""delta.enableInCommitTimestamps"":""true"",""databricks.internal.autoUpgrades.delta.enableInCommitTimestamps"":""true"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-0f56e713-2761-4ea1-b0d3-130bacba3e76""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(2298233917058441),8307133d-07a1-4993-8666-e24865dcdf49,1116-214720-gdpjzh64-v2n,null,WriteSerializable,false,"Map(numFiles -> 4, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 4069, numOutputBytes -> 185991)",null,Databricks-Runtime/19.15.x-aarch64-photon-scala2.13


## Retos ★★★ (si te sobra tiempo)

### Reto 1 · Funciones `extraer()`, `transformar()`, `cargar()` + test

Reorganiza tu ETL en tres funciones y escribe un pequeño **test** de `transformar()` con un DataFrame
de ejemplo hecho a mano. Así podrás reutilizar y probar el código (buena práctica de ingeniería de software).

In [0]:
import pandas as pd

MAPA_PAIS = {
    "ES": "ES", "ESP": "ES", "ESPAÑA": "ES", "SPAIN": "ES",
    "PT": "PT", "PRT": "PT", "PORTUGAL": "PT",
    "GB": "GB", "UK": "GB", "UNITED KINGDOM": "GB", "REINO UNIDO": "GB",
    "US": "US", "USA": "US", "ESTADOS UNIDOS": "US",
}
MAPA_MARKETING = {"SÍ": True, "SI": True, "S": True, "1": True, "TRUE": True,
                  "NO": False, "N": False, "0": False, "FALSE": False}

def extraer(ruta):
    return pd.read_csv(ruta, dtype=str)

def transformar(pdf):
    out = pdf.drop_duplicates().copy()
    out["pais"] = out["pais"].str.strip().str.upper().map(MAPA_PAIS)
    out["email"] = out["email"].str.strip().str.lower()
    out["fecha_alta"] = pd.to_datetime(out["fecha_alta"], format="%Y-%m-%d %H:%M:%S")
    out["acepta_marketing"] = (out["acepta_marketing"]
                               .str.normalize("NFKD")
                               .str.encode("ascii", "ignore").str.decode("ascii")   # "Sí" → "Si"
                               .str.strip().str.upper()
                               .map(MAPA_MARKETING)
                               .astype("boolean"))
    return out

def cargar(pdf, tabla):
    df = spark.createDataFrame(pdf)
    df = df.withColumn("_ingest_ts", F.current_timestamp())
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabla)
    return spark.table(tabla).count()

def test_transformar():
    prueba = pd.DataFrame({
        "pais":             [" españa ", " españa ", "UK"],
        "email":            ["ANA@MAIL.COM ", "ANA@MAIL.COM ", "bob@mail.com"],
        "fecha_alta":       ["2025-01-01 10:00:00"] * 2 + ["2025-02-01 12:00:00"],
        "acepta_marketing": ["Sí", "Sí", None],
    })
    res = transformar(prueba)
    assert len(res) == 2                              # quita el duplicado
    assert list(res["pais"]) == ["ES", "GB"]          # " españa " → ES, UK → GB
    assert res["email"].iloc[0] == "ana@mail.com"     # minúsculas y sin espacios
    assert res["acepta_marketing"].iloc[0] == True    # "Sí" → True
    assert pd.isna(res["acepta_marketing"].iloc[1])   # vacío → nulo
    print("✅ test_transformar OK")

test_transformar()
n = cargar(transformar(extraer(f"{RAW}/clientes/clientes.csv")), "primer_etl_clientes")
print("Filas cargadas:", n)

✅ test_transformar OK
Filas cargadas: 4069


### Reto 2 · Idempotencia: `overwrite` frente a `append`

Ejecuta la carga dos veces con `overwrite` y comprueba en el historial que el número de filas no cambia.
Después, carga **dos veces** en una tabla de pruebas con `mode("append")`. ¿Qué ocurre? ¿Cómo lo arreglarías?
(Pista: en la S5 veremos `MERGE`).

In [0]:
historial = spark.sql("DESCRIBE HISTORY primer_etl_clientes")
display(historial.select("version", "timestamp", "operation",
                         F.col("operationMetrics.numOutputRows").alias("filas_escritas")))

# TODO: escribe dos veces el mismo DataFrame en "primer_etl_clientes_append" con mode("append"),
#       cuenta las filas y explica el resultado. Al acabar, borra la tabla de pruebas.

version,timestamp,operation,filas_escritas
1,2026-11-16T22:11:22.771Z,CREATE OR REPLACE TABLE AS SELECT,4069
0,2026-11-16T21:55:32.367Z,CREATE OR REPLACE TABLE AS SELECT,4069


### Reto 3 · Git folders

1. Crea un repositorio vacío en GitHub (p. ej. `etl-cunef`).
2. En Databricks: **Workspace → Create → Git folder** y pega la URL del repositorio
   (para hacer *push* necesitas un *personal access token* de GitHub en **Settings → Linked accounts**).
3. Mueve o clona este notebook dentro del Git folder, haz **commit & push** desde el botón de Git.

## Para practicar más

- Normaliza también `segmento` (`Particular`/`Empresa`) y `ciudad` (sin espacios sobrantes, con
  `str.title()`; vacía → nulo) y repite la consulta de marketing por segmento.
- Detecta los **40 clientes repetidos** que quedan con el mismo `id_cliente` (versiones antiguas) y quédate
  con la de `fecha_actualizacion` más reciente (`sort_values` + `drop_duplicates(subset=..., keep="last")`).
- Repite el ETL con **PySpark** en lugar de pandas (`spark.read.csv(..., header=True)`); lo veremos en la S2.
- Lectura recomendada: *Fundamentals of Data Engineering*, cap. 1–2 (ciclo de vida de la ingeniería de datos).